# No.3 2D FFT（MRI k空間 ↔ 画像空間）

MRI画像（実空間）と k空間（周波数空間）の相互変換を行います。

$$K(k_x, k_y) = \mathcal{F}_{2D}\{I(x, y)\}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import japanize_matplotlib
import scipy.io
import skimage.io


## データ読み込み

`mri128def.mat` を使います。MATLABの `.mat` ファイルは `scipy.io.loadmat` で読めます。

In [ ]:
data = scipy.io.loadmat('mri128def.mat')
# キー名確認（__header__ 等の内部キーを除外）
keys = [k for k in data.keys() if not k.startswith('_')]
print('利用可能なキー:', keys)

Signal = data[keys[0]].astype(complex)
print(f'Shape: {Signal.shape}, dtype: {Signal.dtype}')

plt.figure(figsize=(5, 5))
plt.imshow(np.abs(Signal), cmap='gray')
plt.title('入力画像（MRI）')
plt.axis('off')
plt.show()

## 2D FFT（画像 → k空間）

**MATLABとの対応:**
```matlab
Output = fftshift(fft2(fftshift(Signal)));
```
```python
Output = np.fft.fftshift(np.fft.fft2(np.fft.fftshift(Signal)))
```

In [ ]:
Kspace = np.fft.fftshift(np.fft.fft2(np.fft.fftshift(Signal)))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(Kspace.real, cmap='gray')
axes[0].set_title('k空間 — Real part')
axes[0].axis('off')
axes[1].imshow(np.log1p(np.abs(Kspace)), cmap='gray')
axes[1].set_title('k空間 — log|magnitude|')
axes[1].axis('off')
fig.tight_layout()
plt.show()

## 2D IFFT（k空間 → 画像へ逆変換）

k空間から元の画像を復元します。

In [ ]:
Reconstructed = np.fft.fftshift(np.fft.ifft2(np.fft.fftshift(Kspace)))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(np.abs(Signal), cmap='gray')
axes[0].set_title('元画像')
axes[0].axis('off')
axes[1].imshow(np.abs(Reconstructed), cmap='gray')
axes[1].set_title('IFFT再構成（元画像と一致するはず）')
axes[1].axis('off')
fig.tight_layout()
plt.show()

print(f'最大誤差: {np.max(np.abs(Signal - Reconstructed)):.2e}')

## 保存

In [ ]:
scipy.io.savemat('mri128def_FFT2D.mat', {'Signal': Kspace})
print('mri128def_FFT2D.mat を保存しました')